# cuplyr against data.table on a free Colab GPU

[cuplyr](https://github.com/bbtheo/cuplyr) runs dplyr code on an NVIDIA GPU through RAPIDS libcudf.
This notebook races it against plain [data.table](https://rdatatable.gitlab.io/data.table/), the package R users
reach for when speed matters, on the same data and tasks. It doesn't use dtplyr: the data.table side is
written in `[i, j, by]` the way an experienced data.table user would write it.

The rules for keeping it fair:

- **data.table gets every CPU core.** Its default is half of them, so `setDTthreads(0)` is set.
- **data.table gets its fast path.** The aggregations compute their columns first and then call
  plain `sum()`, `mean()` and `max()` by group, so data.table's GForce optimisation applies. A cell
  below prints data.table's own verbose output to show it.
- **Both sides must agree.** Every task compares the two results, and the `same_result` column says
  whether they match.
- **The timings are medians** of five runs after one warm-up run (three runs for the file tasks).
- **The upload is not hidden.** The in-memory tasks start from tables already on the GPU, as
  data.table starts from tables already in RAM. The notebook prints how long the copy to the GPU
  took, and the file tasks include all of the reading, so nothing comes for free.

To run it:

1. **Runtime → Change runtime type**, choose **T4 GPU** and keep the runtime type as **Python 3**.
2. **Runtime → Run all.**

The first cell installs everything, which takes 5 to 10 minutes. The races take a few minutes more.
A T4 is a 2018 card and Colab gives you two CPU cores, so the ratios will differ from those on a desktop.


In [ ]:
#@title Install cuplyr and data.table (5 to 10 minutes) { display-mode: "form" }
# Installs R, RAPIDS libcudf, data.table, nanoparquet and cuplyr into /opt/cuplyr, then adds a %%R cell magic
# that sends each cell to one R session kept open for the whole notebook.
import os, subprocess, tempfile, uuid
from IPython.core.magic import register_cell_magic

PREFIX = os.environ.get("CUPLYR_PREFIX", "/opt/cuplyr")
R_BIN = f"{PREFIX}/bin/R"
LOG = "/tmp/cuplyr-install.log"

INSTALL_SH = r'''#!/usr/bin/env bash
# R, libcudf, the CUDA toolkit and the compilers all come from one micromamba env,
# so nothing mixes with the libraries of the host's own R or Python.
set -euo pipefail
PREFIX="${PREFIX:-/opt/cuplyr}"
MAMBA_ROOT="${MAMBA_ROOT:-/opt/micromamba}"
CUPLYR_REF="${CUPLYR_REF:-main}"
SRC="${SRC:-/tmp/cuplyr-src}"
LOG="${LOG:-/tmp/cuplyr-install.log}"
: > "$LOG"

echo "==> GPU: $(nvidia-smi --query-gpu=name,memory.total --format=csv,noheader)"

# libcudf 25.12 is built for CUDA 12; keep the runtime no newer than the driver supports.
drv=$(nvidia-smi | grep -oE 'CUDA (UMD )?Version: [0-9]+\.[0-9]+' | grep -oE '[0-9]+\.[0-9]+' | head -1)
case "$drv" in
  12.*) cuda_pin="cuda-version>=12.2,<=$drv" ;;
  *)    cuda_pin="cuda-version>=12.2,<13" ;;
esac

mm="$MAMBA_ROOT/bin/micromamba"
if [ ! -x "$mm" ]; then
  mkdir -p "$MAMBA_ROOT/bin"
  curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C "$MAMBA_ROOT" bin/micromamba
fi
export MAMBA_ROOT_PREFIX="$MAMBA_ROOT"

if [ ! -f "$PREFIX/include/cudf/types.hpp" ]; then
  echo "==> Downloading R, RAPIDS libcudf 25.12 and the CUDA toolkit ($cuda_pin)"
  "$mm" create -y -p "$PREFIX" -c rapidsai -c conda-forge -c nvidia \
    "libcudf=25.12" "librmm=25.12" "libkvikio=25.12" "$cuda_pin" \
    "cuda-nvcc>=12.2,<13" "cuda-cudart-dev>=12.2,<13" "cuda-driver-dev>=12.2,<13" \
    "r-base>=4.3" r-rcpp r-dplyr r-rlang r-vctrs r-pillar r-glue r-cli r-tidyselect r-tibble "r-lifecycle>=1.0.5" \
    r-data.table r-nanoparquet \
    >> "$LOG" 2>&1
fi

# configure expects $CUDA_HOME/{bin/nvcc,include/cuda.h,lib64}; conda splits those across bin/ and targets/.
# Link against the stub libcuda.so; at run time its soname (libcuda.so.1) resolves to the host driver.
CUDA_HOME="$PREFIX/cuda-home"
mkdir -p "$CUDA_HOME"
ln -sfn "$PREFIX/bin" "$CUDA_HOME/bin"
ln -sfn "$PREFIX/targets/x86_64-linux/include" "$CUDA_HOME/include"
rm -rf "$CUDA_HOME/lib64"; mkdir -p "$CUDA_HOME/lib64"
ln -sf "$PREFIX"/targets/x86_64-linux/lib/libcudart.so* "$CUDA_HOME/lib64/"
ln -sf "$PREFIX/targets/x86_64-linux/lib/stubs/libcuda.so" "$CUDA_HOME/lib64/libcuda.so"

echo "==> Building cuplyr ($CUPLYR_REF) from GitHub"
rm -rf "$SRC"
git clone -q --depth 1 -b "$CUPLYR_REF" https://github.com/bbtheo/cuplyr.git "$SRC"
cd "$SRC"
export CONDA_PREFIX="$PREFIX" CUDA_HOME PATH="$PREFIX/bin:$PATH" MAKEFLAGS="-j$(nproc)"
export R_ENVIRON_USER=/dev/null R_PROFILE_USER=/dev/null R_MAKEVARS_USER=/dev/null
./configure >> "$LOG" 2>&1
"$PREFIX/bin/R" CMD INSTALL --no-docs . >> "$LOG" 2>&1
echo "==> Done"
'''

if not os.path.exists(f"{PREFIX}/lib/R/library/cuplyr"):
    with open("/tmp/install-cuplyr.sh", "w") as f:
        f.write(INSTALL_SH)
    env = dict(os.environ, PREFIX=PREFIX, LOG=LOG)
    proc = subprocess.Popen(["bash", "/tmp/install-cuplyr.sh"], env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        print(open(LOG).read()[-5000:])
        raise RuntimeError(f"Install failed; the full log is in {LOG}")
else:
    print("cuplyr is already installed")

_SENTINEL = "__cell_done_" + uuid.uuid4().hex
_r = None

def _start_r():
    global _r
    _r = subprocess.Popen([R_BIN, "--vanilla", "--quiet", "--no-echo"], text=True, bufsize=1,
                          stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    _r.stdin.write("options(warn = 1, width = 100); sink(stdout(), type = 'message')\n"
                   ".run_cell <- function(f) tryCatch(\n"
                   "  source(f, local = globalenv(), print.eval = TRUE, max.deparse.length = Inf),\n"
                   "  error = function(e) cat('Error: ', conditionMessage(e), '\\n', sep = ''))\n")
    _r.stdin.flush()

def _run(code):
    with tempfile.NamedTemporaryFile("w", suffix=".R", delete=False) as f:
        f.write(code)
    _r.stdin.write(f".run_cell('{f.name}'); cat('\\n{_SENTINEL}\\n'); flush(stdout())\n")
    _r.stdin.flush()
    for line in _r.stdout:
        if line.rstrip("\n") == _SENTINEL:
            break
        print(line, end="", flush=True)
    else:
        raise RuntimeError("R exited; run the cell again to start a new R session")
    os.unlink(f.name)

@register_cell_magic
def R(line, cell):
    if _r is None or _r.poll() is not None:
        _start_r()
    _run(cell)

print("Ready: start a cell with %%R to run R")


Every cell below starts with `%%R`. That line sends the cell to an R session that stays open,
so variables carry over from one cell to the next. Everything else is plain R.


In [ ]:
%%R
library(data.table)
library(dplyr, warn.conflicts = FALSE)
library(cuplyr, warn.conflicts = FALSE)
setDTthreads(0)
cat("data.table", format(packageVersion("data.table")), "using", getDTthreads(), "threads\n")


## The race

`race()` checks that both results match, times each side, and adds a row to `results`.


In [ ]:
%%R
median_secs <- function(f, reps = 5) {
  f()
  median(replicate(reps, system.time(f())[["elapsed"]]))
}
same <- function(a, b, by) {
  a <- as.data.frame(a); b <- as.data.frame(b)[names(a)]
  if (length(by)) { a <- a[do.call(order, a[by]), ]; b <- b[do.call(order, b[by]), ] }
  isTRUE(all.equal(a, b, check.attributes = FALSE, tolerance = 1e-6))
}
results <- data.frame()
race <- function(task, dt_fun, gpu_fun, by = NULL, reps = 5) {
  ok <- same(dt_fun(), gpu_fun(), by)
  t_dt <- median_secs(dt_fun, reps)
  t_gpu <- median_secs(gpu_fun, reps)
  row <- data.frame(task = task, data.table_ms = round(t_dt * 1000), cuplyr_ms = round(t_gpu * 1000),
                    speedup = round(t_dt / t_gpu, 1), same_result = ok)
  results <<- rbind(results[results$task != task, ], row)
  print(row, row.names = FALSE)
  invisible(row)
}


## The data

A sales table with 20 million rows: 500 stores, four channels, 100,000 products and a million customers.
There are also two lookup tables: stores to regions (500 rows) and customers to segments (a million rows).


In [ ]:
%%R
set.seed(42)
n <- 2e7
fact <- data.table(
  store    = sample(500L, n, TRUE),
  channel  = sample(c("web", "shop", "phone", "app"), n, TRUE),
  product  = sample(1e5L, n, TRUE),
  customer = sample(1e6L, n, TRUE),
  price    = round(runif(n, 1, 300), 2),
  qty      = sample(20L, n, TRUE),
  discount = round(runif(n, 0, 0.3), 2)
)
stores    <- data.table(store = 1:500, region = sample(c("north", "south", "east", "west", "central"), 500, TRUE))
customers <- data.table(customer = 1:1e6, segment = sample(c("consumer", "business", "public"), 1e6, TRUE))
upload <- system.time({
  fact_gpu      <- tbl_gpu(fact)
  stores_gpu    <- tbl_gpu(stores)
  customers_gpu <- tbl_gpu(customers)
})[["elapsed"]]
cat(sprintf("%s rows in R: %.0f MB. Copying the tables to the GPU took %.1f s.\n",
            format(n, big.mark = ","), object.size(fact) / 2^20, upload))


## Group by with few groups

Filter on a number and a string, compute revenue, then group by store and channel (2,000 groups).


In [ ]:
%%R
dt_few_on <- function(x) {
  x <- x[price >= 150 & channel != "phone", .(store, channel, price, qty, revenue = price * qty * (1 - discount))]
  x[, .(n = .N, revenue = sum(revenue), avg_price = mean(price), max_qty = max(qty)), by = .(store, channel)]
}
dt_few <- function() dt_few_on(fact)
gpu_few <- function(d = fact_gpu) d |>
  filter(price >= 150, channel != "phone") |>
  mutate(revenue = price * qty * (1 - discount)) |>
  group_by(store, channel) |>
  summarise(n = n(), revenue = sum(revenue), avg_price = mean(price), max_qty = max(qty), .groups = "drop") |>
  collect()
race("group by, 2,000 groups", dt_few, gpu_few, by = c("store", "channel"))


The proof that data.table took its optimised path. Look for `GForce optimized j`:


In [ ]:
%%R
options(datatable.verbose = TRUE)
out <- capture.output(invisible(dt_few()))
options(datatable.verbose = FALSE)
cat(grep("GForce", out, value = TRUE), sep = "\n")


## Group by with a million groups

Every customer becomes a group. A million summary rows come back to R.


In [ ]:
%%R
race("group by, 1M groups",
  function() {
    x <- fact[, .(customer, price, spend = price * qty)]
    x[, .(orders = .N, spend = sum(spend), max_price = max(price)), by = customer]
  },
  function() fact_gpu |>
    mutate(spend = price * qty) |>
    group_by(customer) |>
    summarise(orders = n(), spend = sum(spend), max_price = max(price), .groups = "drop") |>
    collect(),
  by = "customer")


## Joins

First a join to a small table (500 stores), then a join to a big one (a million customers), each
followed by a group by. data.table uses an update join (`:=` with `on =`), which is its fastest form.


In [ ]:
%%R
race("join 500-row table, then group",
  function() {
    x <- fact[price >= 150 & channel != "phone", .(store, channel, revenue = price * qty * (1 - discount))]
    x[stores, on = "store", region := i.region]
    x[, .(n = .N, revenue = sum(revenue)), by = .(region, channel)]
  },
  function() fact_gpu |>
    filter(price >= 150, channel != "phone") |>
    mutate(revenue = price * qty * (1 - discount)) |>
    inner_join(stores_gpu, by = "store") |>
    group_by(region, channel) |>
    summarise(n = n(), revenue = sum(revenue), .groups = "drop") |>
    collect(),
  by = c("region", "channel"))
race("join 1M-row table, then group",
  function() {
    x <- fact[, .(customer, revenue = price * qty)]
    x[customers, on = "customer", segment := i.segment]
    x[, .(n = .N, revenue = sum(revenue)), by = segment]
  },
  function() fact_gpu |>
    inner_join(customers_gpu, by = "customer") |>
    mutate(revenue = price * qty) |>
    group_by(segment) |>
    summarise(n = n(), revenue = sum(revenue), .groups = "drop") |>
    collect(),
  by = "segment")


## Top N

Total revenue for each of 100,000 products, then the top 10.


In [ ]:
%%R
race("top 10 of 100k products",
  function() {
    x <- fact[channel != "phone", .(product, revenue = price * qty)]
    x[, .(revenue = sum(revenue)), by = product][order(-revenue)][1:10]
  },
  function() fact_gpu |>
    filter(channel != "phone") |>
    mutate(revenue = price * qty) |>
    group_by(product) |>
    summarise(revenue = sum(revenue), .groups = "drop") |>
    arrange(desc(revenue)) |>
    head(10) |>
    collect(),
  by = "product")


## Rolling windows

A 50-step rolling mean on each of 2,000 time series (10,000 points each), then the peak and the mean
of each series. data.table uses `frollmean()`, and cuplyr uses `roll_mean()`.


In [ ]:
%%R
series <- data.table(id = rep(1:2000, each = n / 2000), x = cumsum(rnorm(n)))
series_gpu <- tbl_gpu(series)
race("rolling mean, 2,000 series",
  function() {
    r <- series[, .(r = frollmean(x, 50)), by = id]
    r[, .(peak = max(r, na.rm = TRUE), avg = mean(r, na.rm = TRUE)), by = id]
  },
  function() series_gpu |>
    group_by(id) |>
    mutate(r = roll_mean(x, 50)) |>
    summarise(peak = max(r, na.rm = TRUE), avg = mean(r, na.rm = TRUE), .groups = "drop") |>
    collect(),
  by = "id")


## Bringing a big result back

This is the hardest case for cuplyr: the result is 10 million rows, which cuplyr has to copy from GPU memory into R.


In [ ]:
%%R
race("return 10M rows to R",
  function() fact[price >= 150, .(store, product, price, qty)],
  function() fact_gpu |> filter(price >= 150) |> select(store, product, price, qty) |> collect())


## Starting from a file

Real work usually starts from a file. The first race reads parquet on both sides: data.table through
[nanoparquet](https://nanoparquet.r-lib.org/), cuplyr through `read_parquet_gpu()`, which reads only
the needed columns and pushes the filter into the scan. The second race gives data.table `fread()`
on a CSV file instead, its best-known strength. The time includes the read.


In [ ]:
%%R
pq  <- "/tmp/fact.parquet"
csv <- "/tmp/fact.csv"
nanoparquet::write_parquet(fact, pq)
fwrite(fact, csv)
cat(sprintf("parquet %.0f MB, csv %.0f MB\n", file.size(pq) / 2^20, file.size(csv) / 2^20))
cols <- c("store", "channel", "price", "qty", "discount")
race("parquet file to summary",
  function() dt_few_on(setDT(nanoparquet::read_parquet(pq, col_select = cols))),
  function() gpu_few(read_parquet_gpu(pq)),
  by = c("store", "channel"), reps = 3)
race("csv (fread) vs parquet (GPU)",
  function() dt_few_on(fread(csv, select = cols)),
  function() gpu_few(read_parquet_gpu(pq)),
  by = c("store", "channel"), reps = 3)


## Where data.table catches up

The first pipeline on growing slices of the table. On small data a GPU's fixed costs (launching
kernels, copying the result back) are as large as the work itself, and data.table keeps up.


In [ ]:
%%R
for (m in c(1e4, 1e5, 1e6, 1e7)) {
  small <- fact[seq_len(m)]
  small_gpu <- tbl_gpu(small)
  race(sprintf("group by, %s rows", format(m, big.mark = ",", scientific = FALSE)),
    function() dt_few_on(small), function() gpu_few(small_gpu), by = c("store", "channel"))
}
rm(small, small_gpu)


## All results


In [ ]:
%%R
print(results, row.names = FALSE)


## Your turn

Change a task and run it again. If you find one where cuplyr gives a different answer or falls
over, please [open an issue](https://github.com/bbtheo/cuplyr/issues).
